# Maqam labels — Plotly analysis

Loaded from the completed `maqam-167-v1` run: **163 labels** (25 withheld).
These are weak labels from 12-TET ABC input; there is no ground truth.

The full self-contained dashboard is `exports/plots/index.html`.
Run `python3 scripts/plot_labels.py` to regenerate everything.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('exports/plots/label_data.csv')
print(len(df), 'labels')
df[['artist','maqam','maqam_status','tonic','tonic_confidence','overall_confidence','fit']].head()


In [ ]:
summary = (df.groupby(['artist','maqam'], dropna=False)
             .size().reset_index(name='songs')
             .sort_values(['artist','songs'], ascending=[True,False]))
summary


In [ ]:
fig = px.bar(
    df, x='maqam', color='maqam_status',
    category_orders={'maqam_status':['consistent','possible','withheld']},
    title='Maqam label distribution (colored by status)',
    labels={'maqam':'label','count':'songs'},
)
fig.update_layout(barmode='stack', template='plotly_white', height=480)
fig.show()


In [ ]:
fig = px.histogram(
    df, x='overall_confidence', color='maqam_status',
    category_orders={'overall_confidence':['low','medium','high'],
                     'maqam_status':['consistent','possible','withheld']},
    barmode='group', title='Overall confidence by label status',
    template='plotly_white', height=440,
)
fig.show()


In [ ]:
fig = px.scatter(
    df, x='tonic_margin', y='fit', color='maqam', symbol='maqam_status',
    hover_name='title', hover_data=['artist','tonic','overall_confidence','flags'],
    title='Evidence: scale fit vs tonic margin (size = number of flags)',
    size='n_flags', size_max=18, template='plotly_white', height=600,
)
fig.add_hline(y=0.90, line_dash='dash', line_color='#D1495B',
              annotation_text='fit 0.90: no single scale below this')
fig.show()


In [ ]:
share = (pd.crosstab(df['artist'], df['maqam'], normalize='index') * 100).round(1)
fig = px.bar(
    share, barmode='group', title='Maqam share by artist (%)',
    template='plotly_white', height=480,
)
fig.update_layout(yaxis_title='% of labels', xaxis_title='label')
fig.show()


In [ ]:
drift = df.assign(sections_disagree=df['section_tonics'] > 1)
fig = px.histogram(
    drift, x='artist', color='sections_disagree', barmode='group',
    title='Songs whose sections disagree on the top tonic',
    template='plotly_white', height=440,
)
fig.show()


## Full dashboard

Open `exports/plots/index.html` for all views:

- overview, artist/album breakdowns, tonic × maqam heatmap
- fit vs margin, section drift, confidence flow, duplicate-title consistency

Caveats: Nahawand can be a rounded Bayati; Kurd can be a rounded Bayati; Ajam
can be a rounded Rast. The 25 withheld labels are cases where the analyzer or
agent declined a scale. The 4 no-ABC tracks are not plotted.
